# FastAPI na Prática — Bella Tavola 🍝

## BLOCO 1


### Exercício 1.1 — A porta de entrada

In [ ]:
!pip install fastapi uvicorn pydantic-settings

In [ ]:
from fastapi import FastAPI

app = FastAPI(
    title="Bella Tavola API",
    description="API do restaurante Bella Tavola",
    version="1.0.0"
)

@app.get("/")
async def root():
    return {
        "restaurante": "Bella Tavola",
        "mensagem": "Bem-vindo à nossa API",
        "chef": "Henrique Fogaça",
        "cidade": "São Paulo",
        "especialidade": "Cozinha Brasileira"
    }

### Exercício 1.2 — O cardápio completo

In [ ]:
pratos = [
    {"id": 1, "nome": "Pizza Portuguesa", "categoria": "pizza", "preco": 59.9},
    {"id": 2, "nome": "Pizza Calabresa", "categoria": "pizza", "preco": 54.9},
    {"id": 3, "nome": "Espaguete ao Alho e Óleo", "categoria": "massa", "preco": 39.9},
    {"id": 4, "nome": "Nhoque ao Sugo", "categoria": "massa", "preco": 44.9},
    {"id": 5, "nome": "Torta de Limão", "categoria": "sobremesa", "preco": 18.9},
    {"id": 6, "nome": "Pudim de Leite", "categoria": "sobremesa", "preco": 16.9},
]

@app.get("/pratos")
async def listar_pratos():
    return pratos

### Exercício 1.3 — Buscando um prato específico

In [ ]:
@app.get("/pratos/{prato_id}")
async def buscar_prato(prato_id: int):
    for prato in pratos:
        if prato["id"] == prato_id:
            return prato
    return {"mensagem": "Prato não encontrado"}


# devolver status 200 com mensagem de erro é ruim porque o cliente da API
# entende que a operação deu certo, quando na verdade o recurso não existe.

### Exercício 1.4 — Filtrando o cardápio

In [ ]:
from typing import Optional

@app.get("/pratos")
async def listar_pratos(
    categoria: Optional[str] = None,
    preco_maximo: Optional[float] = None
):
    resultado = pratos

    if categoria:
        resultado = [p for p in resultado if p["categoria"] == categoria]

    if preco_maximo is not None:
        resultado = [p for p in resultado if p["preco"] <= preco_maximo]

    return resultado

### Exercício 1.5 — Path e query juntos


In [ ]:
from typing import Optional

pratos = [
    {"id": 1, "nome": "Pizza Portuguesa", "categoria": "pizza", "preco": 59.9, "disponivel": True},
    {"id": 2, "nome": "Pizza Calabresa", "categoria": "pizza", "preco": 54.9, "disponivel": True},
    {"id": 3, "nome": "Espaguete ao Alho e Óleo", "categoria": "massa", "preco": 39.9, "disponivel": True},
    {"id": 4, "nome": "Nhoque ao Sugo", "categoria": "massa", "preco": 44.9, "disponivel": False},
    {"id": 5, "nome": "Torta de Limão", "categoria": "sobremesa", "preco": 18.9, "disponivel": True},
    {"id": 6, "nome": "Pudim de Leite", "categoria": "sobremesa", "preco": 16.9, "disponivel": True},
]

@app.get("/pratos/{prato_id}")
async def buscar_prato(prato_id: int, formato: str = "completo"):
    for prato in pratos:
        if prato["id"] == prato_id:
            if formato == "resumido":
                return {"nome": prato["nome"], "preco": prato["preco"]}
            return prato
    return {"mensagem": "Prato não encontrado"}

@app.get("/pratos")
async def listar_pratos(
    categoria: Optional[str] = None,
    preco_maximo: Optional[float] = None,
    apenas_disponiveis: bool = False
):
    resultado = pratos
    if categoria:
        resultado = [p for p in resultado if p["categoria"] == categoria]
    if preco_maximo is not None:
        resultado = [p for p in resultado if p["preco"] <= preco_maximo]
    if apenas_disponiveis:
        resultado = [p for p in resultado if p["disponivel"]]
    return resultado

### Exercício 1.6 — Cadastrando um prato


In [ ]:
from pydantic import BaseModel
from typing import Optional

class PratoInput(BaseModel):
    nome: str
    categoria: str
    preco: float
    descricao: Optional[str] = None
    disponivel: bool = True

@app.post("/pratos")
async def criar_prato(prato: PratoInput):
    novo_id = max(p["id"] for p in pratos) + 1
    novo_prato = {"id": novo_id, **prato.model_dump()}
    pratos.append(novo_prato)
    return novo_prato

# quando o servidor reinicia, os dados são perdidos,
# porque a lista está só na memória RAM do processo.

### Exercício 1.7 — Separando entrada e saída


In [ ]:
from datetime import datetime
from pydantic import BaseModel
from typing import Optional

class PratoInput(BaseModel):
    nome: str
    categoria: str
    preco: float
    descricao: Optional[str] = None
    disponivel: bool = True

class PratoOutput(BaseModel):
    id: int
    nome: str
    categoria: str
    preco: float
    descricao: Optional[str]
    disponivel: bool
    criado_em: str

@app.post("/pratos", response_model=PratoOutput)
async def criar_prato(prato: PratoInput):
    novo_id = max(p["id"] for p in pratos) + 1
    novo_prato = {
        "id": novo_id,
        "criado_em": datetime.now().isoformat(),
        **prato.model_dump()
    }
    pratos.append(novo_prato)
    return novo_prato

### Exercício 1.8 — Desafio: as bebidas 🍷

In [ ]:
from datetime import datetime
from typing import Optional
from pydantic import BaseModel

bebidas = [
    {"id": 1, "nome": "Água Mineral", "tipo": "agua", "preco": 6.0, "alcoolica": False, "volume_ml": 500, "criado_em": "2026-04-20T12:00:00"},
    {"id": 2, "nome": "Coca Cola", "tipo": "refrigerante", "preco": 8.0, "alcoolica": False, "volume_ml": 350, "criado_em": "2026-04-20T12:00:00"},
    {"id": 3, "nome": "Suco de Laranja", "tipo": "suco", "preco": 10.0, "alcoolica": False, "volume_ml": 300, "criado_em": "2026-04-20T12:00:00"},
    {"id": 4, "nome": "Cerveja Heineken", "tipo": "cerveja", "preco": 14.0, "alcoolica": True, "volume_ml": 600, "criado_em": "2026-04-20T12:00:00"},
    {"id": 5, "nome": "Chá Gelado", "tipo": "cha", "preco": 9.0, "alcoolica": False, "volume_ml": 300, "criado_em": "2026-04-20T12:00:00"},
]

class BebidaInput(BaseModel):
    nome: str
    tipo: str
    preco: float
    alcoolica: bool
    volume_ml: int

class BebidaOutput(BaseModel):
    id: int
    nome: str
    tipo: str
    preco: float
    alcoolica: bool
    volume_ml: int
    criado_em: str

@app.get("/bebidas")
async def listar_bebidas(
    tipo: Optional[str] = None,
    alcoolica: Optional[bool] = None
):
    resultado = bebidas
    if tipo:
        resultado = [b for b in resultado if b["tipo"] == tipo]
    if alcoolica is not None:
        resultado = [b for b in resultado if b["alcoolica"] == alcoolica]
    return resultado

@app.get("/bebidas/{bebida_id}")
async def buscar_bebida(bebida_id: int):
    for bebida in bebidas:
        if bebida["id"] == bebida_id:
            return bebida
    return {"mensagem": "Bebida não encontrada"}

@app.post("/bebidas", response_model=BebidaOutput)
async def criar_bebida(bebida: BebidaInput):
    novo_id = max(b["id"] for b in bebidas) + 1
    nova_bebida = {
        "id": novo_id,
        "criado_em": datetime.now().isoformat(),
        **bebida.model_dump()
    }
    bebidas.append(nova_bebida)
    return nova_bebida

## BLOCO 2


### Exercício 2.1 — Identificando o problema

In [ ]:
# Caso 1 — GET /pratos/99
# Status retornado: 200
# Status correto: 404
# Problema: a API devolve sucesso para um prato que não existe.

# Caso 2 — POST com dados inválidos
# Status retornado: 200
# Status correto: 422
# Problema: sem validação, nome vazio e preço negativo são aceitos.

# Caso 3 — POST com campos faltando
# Status retornado: 422
# Status correto: 422
# Problema: aqui o FastAPI/Pydantic já faz a validação corretamente.

# Caso 4 — GET com tipo errado
# Status retornado: 422
# Status correto: 422
# Problema: aqui o FastAPI já valida o tipo do query parameter.

### Exercício 2.2 — Corrigindo o 404


In [ ]:
from fastapi import HTTPException

@app.get("/pratos/{prato_id}")
async def buscar_prato(prato_id: int, formato: str = "completo"):
    for prato in pratos:
        if prato["id"] == prato_id:
            if formato == "resumido":
                return {"nome": prato["nome"], "preco": prato["preco"]}
            return prato
    raise HTTPException(status_code=404, detail=f"Prato com id {prato_id} não encontrado")

@app.get("/bebidas/{bebida_id}")
async def buscar_bebida(bebida_id: int):
    for bebida in bebidas:
        if bebida["id"] == bebida_id:
            return bebida
    raise HTTPException(status_code=404, detail=f"Bebida com id {bebida_id} não encontrada")

# Usamos raise porque HTTPException é uma exceção.
# Ela interrompe a execução e entrega o erro ao FastAPI com o status correto.

### Exercício 2.3 — Validando com Field


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class PratoInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100)
    categoria: str = Field(pattern="^(pizza|massa|sobremesa|entrada|salada)$")
    preco: float = Field(gt=0)
    descricao: Optional[str] = Field(default=None, max_length=500)
    disponivel: bool = True

class BebidaInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100)
    tipo: str = Field(pattern="^(vinho|agua|refrigerante|suco|cerveja|cha)$")
    preco: float = Field(gt=0)
    alcoolica: bool
    volume_ml: int = Field(ge=50, le=2000)

### Exercício 2.4 — Regras de negócio com validadores


In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional

class PratoInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100)
    categoria: str = Field(pattern="^(pizza|massa|sobremesa|entrada|salada)$")
    preco: float = Field(gt=0)
    preco_promocional: Optional[float] = Field(default=None, gt=0)
    descricao: Optional[str] = Field(default=None, max_length=500)
    disponivel: bool = True

    @field_validator("preco_promocional")
    @classmethod
    def validar_preco_promocional(cls, value, info):
        if value is None:
            return value

        preco_original = info.data.get("preco")
        if preco_original is None:
            return value

        if value >= preco_original:
            raise ValueError("Preço promocional deve ser menor que o preço original")

        desconto = (preco_original - value) / preco_original
        if desconto > 0.5:
            raise ValueError("Desconto não pode ser maior que 50% do preço original")

        return value

### Exercício 2.5 — Múltiplos erros em uma rota


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class DisponibilidadeInput(BaseModel):
    disponivel: bool

@app.put("/pratos/{prato_id}/disponibilidade")
async def alterar_disponibilidade(prato_id: int, body: DisponibilidadeInput):
    for prato in pratos:
        if prato["id"] == prato_id:
            prato["disponivel"] = body.disponivel
            return prato
    raise HTTPException(status_code=404, detail="Prato não encontrado")

pedidos = []

class PedidoInput(BaseModel):
    prato_id: int
    quantidade: int = Field(ge=1)
    observacao: Optional[str] = None

class PedidoOutput(BaseModel):
    id: int
    prato_id: int
    nome_prato: str
    quantidade: int
    valor_total: float
    observacao: Optional[str]

@app.post("/pedidos", response_model=PedidoOutput)
async def criar_pedido(pedido: PedidoInput):
    prato = next((p for p in pratos if p["id"] == pedido.prato_id), None)

    if not prato:
        raise HTTPException(status_code=404, detail="Prato não encontrado")

    if not prato["disponivel"]:
        raise HTTPException(
            status_code=400,
            detail=f"O prato '{prato['nome']}' não está disponível no momento"
        )

    novo_pedido = {
        "id": len(pedidos) + 1,
        "prato_id": pedido.prato_id,
        "nome_prato": prato["nome"],
        "quantidade": pedido.quantidade,
        "valor_total": prato["preco"] * pedido.quantidade,
        "observacao": pedido.observacao
    }
    pedidos.append(novo_pedido)
    return novo_pedido

### Exercício 2.6 — Exception handler global


In [ ]:
from fastapi import Request, HTTPException
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse

@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    return JSONResponse(
        status_code=422,
        content={
            "erro": "Dados de entrada inválidos",
            "status": 422,
            "path": str(request.url),
            "detalhes": [
                {
                    "campo": " -> ".join(str(loc) for loc in e["loc"]),
                    "mensagem": e["msg"]
                }
                for e in exc.errors()
            ]
        }
    )

@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    return JSONResponse(
        status_code=exc.status_code,
        content={
            "erro": exc.detail,
            "status": exc.status_code,
            "path": str(request.url),
            "detalhes": []
        }
    )

### Exercício 2.7 — Desafio: API sob pressão

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI()

reservas = [
    {"id": 1, "mesa": 5, "nome": "Silva", "pessoas": 4, "ativa": True},
    {"id": 2, "mesa": 3, "nome": "Costa", "pessoas": 2, "ativa": False},
]

class ReservaInput(BaseModel):
    mesa: int = Field(ge=1, le=20)           # corrige problema 2
    nome: str = Field(min_length=2, max_length=100)
    pessoas: int = Field(ge=1, le=20)

@app.get("/reservas/{reserva_id}")
async def buscar_reserva(reserva_id: int):
    for r in reservas:
        if r["id"] == reserva_id:
            return r
    raise HTTPException(status_code=404, detail="Reserva não encontrada")  # corrige problema 1

@app.post("/reservas")
async def criar_reserva(reserva: ReservaInput):
    mesa_ocupada = any(                        # corrige problema 5
        r["mesa"] == reserva.mesa and r["ativa"]
        for r in reservas
    )
    if mesa_ocupada:
        raise HTTPException(
            status_code=400,
            detail=f"Mesa {reserva.mesa} já está reservada"
        )
    nova = {"id": len(reservas) + 1, **reserva.model_dump(), "ativa": True}
    reservas.append(nova)
    return nova

@app.delete("/reservas/{reserva_id}")
async def cancelar_reserva(reserva_id: int):
    for r in reservas:
        if r["id"] == reserva_id:
            r["ativa"] = False
            return {"mensagem": "Reserva cancelada com sucesso"}
    raise HTTPException(status_code=404, detail="Reserva não encontrada")  # corrige problema 3

@app.get("/reservas")
async def listar_reservas(apenas_ativas: bool = False):
    if apenas_ativas:
        return [r for r in reservas if r["ativa"]]  # corrige problema 4
    return reservas

In [ ]:
# Problema 1: GET /reservas/{id} retorna 200 com {"erro": ...} em vez de 404.
# Problema 2: POST /reservas não valida regras de negócio, como mesa e quantidade de pessoas.
# Problema 3: DELETE /reservas/{id} não trata o caso de reserva inexistente com HTTPException 404.
# Problema 4: O filtro apenas_ativas compara com a string "true" em vez do boolean True.
# Problema 5: Falta validação para impedir criação de reservas inválidas ou cancelamento inconsistente.

## BLOCO 3

### Exercício 3.1 — Sentindo o problema

In [ ]:
# 1. routers/promocoes.py.

# 2. Chance de conflito de merge no Git.

# 3. Com routers separados, bastaria abrir routers/reservas.py.

### Exercício 3.2 — Extraindo o primeiro router


In [ ]:
# Estrutura esperada:
# routers/__init__.py
# routers/pratos.py
# main.py

# Exemplo resumido:

# routers/pratos.py
from fastapi import APIRouter, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
from datetime import datetime
from routers import pratos

router = APIRouter()

pratos = [
    {"id": 1, "nome": "Pizza Portuguesa", "categoria": "pizza", "preco": 59.9, "disponivel": True, "criado_em": "2026-04-20T12:00:00"},
    {"id": 2, "nome": "Espaguete ao Alho e Óleo", "categoria": "massa", "preco": 39.9, "disponivel": True, "criado_em": "2026-04-20T12:00:00"},
]

class PratoInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100)
    categoria: str = Field(pattern="^(pizza|massa|sobremesa|entrada|salada)$")
    preco: float = Field(gt=0)
    disponivel: bool = True

class PratoOutput(BaseModel):
    id: int
    nome: str
    categoria: str
    preco: float
    disponivel: bool
    criado_em: str

@router.get("/")
async def listar_pratos(categoria: Optional[str] = None):
    if categoria:
        return [p for p in pratos if p["categoria"] == categoria]
    return pratos

@router.get("/{prato_id}")
async def buscar_prato(prato_id: int):
    for prato in pratos:
        if prato["id"] == prato_id:
            return prato
    raise HTTPException(status_code=404, detail="Prato não encontrado")

@router.post("/", response_model=PratoOutput)
async def criar_prato(prato: PratoInput):
    novo_prato = {
        "id": max(p["id"] for p in pratos) + 1,
        "criado_em": datetime.now().isoformat(),
        **prato.model_dump()
    }
    pratos.append(novo_prato)
    return novo_prato

# main.py
from fastapi import FastAPI
from routers import pratos

app = FastAPI(title="Bella Tavola API")
app.include_router(pratos.router, prefix="/pratos", tags=["Pratos"])

### Exercício 3.3 — Separando todos os domínios


In [ ]:
from fastapi import FastAPI, Request, HTTPException
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from routers import pratos, bebidas, pedidos, reservas

app = FastAPI(
    title="Restaurante do Fogaça API",
    description="API didática do restaurante do Fogaça",
    version="1.0.0"
)

@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    return JSONResponse(
        status_code=422,
        content={
            "erro": "Dados de entrada inválidos",
            "status": 422,
            "path": str(request.url),
            "detalhes": [{"campo": " -> ".join(str(loc) for loc in e["loc"]), "mensagem": e["msg"]} for e in exc.errors()]
        }
    )

@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    return JSONResponse(
        status_code=exc.status_code,
        content={"erro": exc.detail, "status": exc.status_code, "path": str(request.url), "detalhes": []}
    )

app.include_router(pratos.router,   prefix="/pratos",   tags=["Pratos"])
app.include_router(bebidas.router,  prefix="/bebidas",  tags=["Bebidas"])
app.include_router(pedidos.router,  prefix="/pedidos",  tags=["Pedidos"])
app.include_router(reservas.router, prefix="/reservas", tags=["Reservas"])

@app.get("/", tags=["Geral"])
async def root():
    return {"restaurante": "Bella Tavola", "versao": "1.0.0"}

### Exercício 3.4 — Separando os modelos

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class PratoInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100)
    categoria: str = Field(pattern="^(pizza|massa|sobremesa|entrada|salada)$")
    preco: float = Field(gt=0)
    preco_promocional: Optional[float] = Field(default=None, gt=0)
    descricao: Optional[str] = Field(default=None, max_length=500)
    disponivel: bool = True

class PratoOutput(BaseModel):
    id: int
    nome: str
    categoria: str
    preco: float
    preco_promocional: Optional[float]
    descricao: Optional[str]
    disponivel: bool
    criado_em: str


### Exercício 3.5 — Configurações com BaseSettings


In [ ]:
!pip install pydantic-settings

In [ ]:
# config.py
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    app_name: str = "Bella Tavola API"
    app_version: str = "1.0.0"
    app_description: str = "API do restaurante Bella Tavola"
    debug: bool = False
    max_mesas: int = 20
    max_pessoas_por_mesa: int = 8

    class Config:
        env_file = ".env"

settings = Settings()

# .env
# APP_NAME=Bella Tavola API
# APP_VERSION=1.0.0
# DEBUG=true
# MAX_MESAS=15
# MAX_PESSOAS_POR_MESA=6

# main.py
from config import settings

app = FastAPI(
    title=settings.app_name,
    version=settings.app_version,
    description=settings.app_description
)

# Exemplo de uso nos modelos:
from pydantic import BaseModel, Field
from config import settings

class ReservaInput(BaseModel):
    mesa: int = Field(ge=1, le=settings.max_mesas)
    nome: str = Field(min_length=2, max_length=100)
    pessoas: int = Field(ge=1, le=settings.max_pessoas_por_mesa)

### Exercício 3.6 — Desafio final: reservas antecipadas 🏆


In [ ]:
# models/reserva.py
from pydantic import BaseModel, Field, field_validator
from datetime import datetime

class ReservaInput(BaseModel):
    mesa: int = Field(ge=1, le=20)
    nome: str = Field(min_length=2, max_length=100)
    pessoas: int = Field(ge=1, le=8)
    data_hora: datetime

    @field_validator("data_hora")
    @classmethod
    def validar_antecedencia(cls, value):
        agora = datetime.now(tz=value.tzinfo)
        if (value - agora).total_seconds() < 3600:
            raise ValueError("Reserva deve ser feita com pelo menos 1 hora de antecedência")
        return value

class ReservaOutput(BaseModel):
    id: int
    mesa: int
    nome: str
    pessoas: int
    data_hora: str
    ativa: bool
    criada_em: str

# routers/reservas.py
from fastapi import APIRouter, HTTPException
from typing import Optional
from datetime import datetime

router = APIRouter()
reservas = []

@router.post("/", response_model=ReservaOutput)
async def criar_reserva(reserva: ReservaInput):
    data_reserva = reserva.data_hora.date()
    conflito = any(
        r["mesa"] == reserva.mesa
        and r["ativa"]
        and datetime.fromisoformat(r["data_hora"]).date() == data_reserva
        for r in reservas
    )
    if conflito:
        raise HTTPException(status_code=400, detail=f"Mesa {reserva.mesa} já está reservada para {data_reserva}")

    nova = {
        "id": len(reservas) + 1,
        "mesa": reserva.mesa,
        "nome": reserva.nome,
        "pessoas": reserva.pessoas,
        "data_hora": reserva.data_hora.isoformat(),
        "ativa": True,
        "criada_em": datetime.now().isoformat()
    }
    reservas.append(nova)
    return nova

@router.get("/")
async def listar_reservas(data: Optional[str] = None, apenas_ativas: bool = True):
    resultado = reservas
    if apenas_ativas:
        resultado = [r for r in resultado if r["ativa"]]
    if data:
        resultado = [
            r for r in resultado
            if datetime.fromisoformat(r["data_hora"]).date().isoformat() == data
        ]
    return resultado

@router.get("/mesa/{numero}")
async def reservas_por_mesa(numero: int):
    return [r for r in reservas if r["mesa"] == numero]

@router.get("/{reserva_id}", response_model=ReservaOutput)
async def buscar_reserva(reserva_id: int):
    for reserva in reservas:
        if reserva["id"] == reserva_id:
            return reserva
    raise HTTPException(status_code=404, detail="Reserva não encontrada")

@router.delete("/{reserva_id}")
async def cancelar_reserva(reserva_id: int):
    for reserva in reservas:
        if reserva["id"] == reserva_id:
            if not reserva["ativa"]:
                raise HTTPException(status_code=400, detail="Reserva já está cancelada")
            reserva["ativa"] = False
            return {"mensagem": "Reserva cancelada com sucesso"}
    raise HTTPException(status_code=404, detail="Reserva não encontrada")